In [1]:
import pandas as pd
import os

BASE_PATH = ".."
RAW_PATH = os.path.join(BASE_PATH, "data", "raw")
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")

TRANS_PATH = os.path.join(PROCESSED_PATH, "transactions.csv")
ACCOUNTS_PATH = os.path.join(PROCESSED_PATH, "accounts.csv")
BANKS_PATH = os.path.join(PROCESSED_PATH, "banks.csv")

trans_df = pd.read_csv(TRANS_PATH)
accounts_df = pd.read_csv(ACCOUNTS_PATH)
banks_df = pd.read_csv(BANKS_PATH)

print("Transactions:", trans_df.shape)
print("Accounts:", accounts_df.shape)
print("Banks:", banks_df.shape)

Transactions: (6924049, 12)
Accounts: (712688, 6)
Banks: (41815, 10)


In [2]:
# Create Account ID for transaction sender
trans_df["Sender Account ID"] = (
    trans_df["From Bank"].astype(str)
    + "_"
    + trans_df["Account"].astype(str)
)

# Create Account ID for transaction receiver
trans_df["Receiver Account ID"] = (
    trans_df["To Bank"].astype(str)
    + "_"
    + trans_df["Account.1"].astype(str)
)

# Bank IDs
trans_df["Sender Bank ID"] = trans_df["From Bank"]
trans_df["Receiver Bank ID"] = trans_df["To Bank"]

# Basic transaction features
trans_df["Is Currency Same"] = (
    trans_df["Receiving Currency"] == trans_df["Payment Currency"]
)

timestamp = pd.to_datetime(
    trans_df["Timestamp"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

trans_df["Transaction Hour"] = timestamp.dt.hour
trans_df["Day of Week"] = timestamp.dt.day_name()

# Get bank country proxy
bank_country = banks_df[
    ["Bank ID", "Country of Top Currency"]
].drop_duplicates("Bank ID")

sender_country = bank_country.rename(columns={
    "Bank ID": "Sender Bank ID",
    "Country of Top Currency": "Sender Bank Country"
})

receiver_country = bank_country.rename(columns={
    "Bank ID": "Receiver Bank ID",
    "Country of Top Currency": "Receiver Bank Country"
})

trans_df = trans_df.merge(
    sender_country,
    on="Sender Bank ID",
    how="left"
)

trans_df = trans_df.merge(
    receiver_country,
    on="Receiver Bank ID",
    how="left"
)

print(trans_df[
    [
        "Transaction ID",
        "Sender Account ID",
        "Receiver Account ID",
        "Sender Bank ID",
        "Receiver Bank ID",
        "Sender Bank Country",
        "Receiver Bank Country"
    ]
].head())

   Transaction ID Sender Account ID Receiver Account ID  Sender Bank ID  \
0               1      11_8000ECA90        11_8000ECA90              11   
1               2    3402_80021DAD0      3402_80021DAD0            3402   
2               3      11_8000ECA90      1120_8006AA910              11   
3               4    3814_8006AD080      3814_8006AD080            3814   
4               5      20_8006AD530        20_8006AD530              20   

   Receiver Bank ID Sender Bank Country Receiver Bank Country  
0                11       United States         United States  
1              3402       United States         United States  
2              1120       United States         United States  
3              3814       United States         United States  
4                20       United States         United States  


In [3]:
# Cross-border indicator
trans_df["Is Cross Border"] = (
    trans_df["Sender Bank Country"]
    != trans_df["Receiver Bank Country"]
)

# If either country is unknown, do not treat it as confirmed cross-border
unknown_country = (
    trans_df["Sender Bank Country"].isna()
    | trans_df["Receiver Bank Country"].isna()
    | (trans_df["Sender Bank Country"] == "Unknown")
    | (trans_df["Receiver Bank Country"] == "Unknown")
)

trans_df.loc[unknown_country, "Is Cross Border"] = False

# Country pair
trans_df["Country Pair"] = (
    trans_df["Sender Bank Country"].fillna("Unknown")
    + " -> "
    + trans_df["Receiver Bank Country"].fillna("Unknown")
)

# Final feature dataset
transaction_features_df = trans_df[
    [
        "Transaction ID",
        "Sender Account ID",
        "Receiver Account ID",
        "Sender Bank ID",
        "Receiver Bank ID",
        "Is Currency Same",
        "Transaction Hour",
        "Day of Week",
        "Sender Bank Country",
        "Receiver Bank Country",
        "Is Cross Border",
        "Country Pair"
    ]
].copy()

print(transaction_features_df.head(10))

   Transaction ID Sender Account ID Receiver Account ID  Sender Bank ID  \
0               1      11_8000ECA90        11_8000ECA90              11   
1               2    3402_80021DAD0      3402_80021DAD0            3402   
2               3      11_8000ECA90      1120_8006AA910              11   
3               4    3814_8006AD080      3814_8006AD080            3814   
4               5      20_8006AD530        20_8006AD530              20   
5               6      12_8006ADD30        12_8006ADD30              12   
6               7      11_800059120      1217_8006AD4E0              11   
7               8      11_8000ECA90        11_8000ECA90              11   
8               9    1120_8006AA910    243166_81470DCF0            1120   
9              10    1217_8006AD4E0      1217_8006AD4E0            1217   

   Receiver Bank ID  Is Currency Same  Transaction Hour Day of Week  \
0                11              True               NaN         NaN   
1              3402             

In [4]:
expected_columns = [
    "Transaction ID",
    "Sender Account ID",
    "Receiver Account ID",
    "Sender Bank ID",
    "Receiver Bank ID",
    "Is Currency Same",
    "Transaction Hour",
    "Day of Week",
    "Sender Bank Country",
    "Receiver Bank Country",
    "Is Cross Border",
    "Country Pair"
]

print("Shape:", transaction_features_df.shape)

print("\nColumns:")
print(transaction_features_df.columns.tolist())

print("\nMissing values:")
print(transaction_features_df.isnull().sum())

print("\nCross-border transactions:")
print(transaction_features_df["Is Cross Border"].value_counts())

assert transaction_features_df.columns.tolist() == expected_columns
assert len(transaction_features_df) == len(trans_df)

OUTPUT_PATH = os.path.join(
    PROCESSED_PATH,
    "transaction_features.csv"
)

transaction_features_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved:", OUTPUT_PATH)

Shape: (6924049, 12)

Columns:
['Transaction ID', 'Sender Account ID', 'Receiver Account ID', 'Sender Bank ID', 'Receiver Bank ID', 'Is Currency Same', 'Transaction Hour', 'Day of Week', 'Sender Bank Country', 'Receiver Bank Country', 'Is Cross Border', 'Country Pair']

Missing values:
Transaction ID                 0
Sender Account ID              0
Receiver Account ID            0
Sender Bank ID                 0
Receiver Bank ID               0
Is Currency Same               0
Transaction Hour         6924049
Day of Week              6924049
Sender Bank Country            0
Receiver Bank Country          0
Is Cross Border                0
Country Pair                   0
dtype: int64

Cross-border transactions:
Is Cross Border
False    6175747
True      748302
Name: count, dtype: int64

Saved: ..\data\processed\transaction_features.csv
